In [1]:
import pandas as pd
import requests
import time
import numpy as np
from urllib.parse import urlencode

def get_adress_data(adress_file):
    if adress_file[-3:]=='csv':
        return pd.read_csv(adress_file)
    if adress_file[-3:]=='ods':
        return pd.read_excel(adress_file, engine="odf")
    print('error: no valid adress file!')
    return pd.DataFrame()

import requests
import time

def geocode_address_overpass(street, housenumber, city, country, pause=1):
    """
    Geocode an address using the Overpass API with country information.

    Parameters:
        street (str): Street name.
        housenumber (str/int): House number.
        city (str): City name.
        country (str): Country name.
        pause (int): Seconds to wait between requests to respect rate limits.

    Returns:
        tuple: (latitude, longitude) if found, else (None, None)
    """
    overpass_url = "https://overpass-api.de/api/interpreter"
    
    # Construct Overpass QL query
    # Attempting to narrow down the search by city and country
    query = f"""
    [out:json];
    area["name"="{city}"]["boundary"="administrative"]["admin_level"~"^(8|9)$"]["is_in:country"="{country}"];
    node
      ["addr:housenumber"="{housenumber}"]
      ["addr:street"="{street}"]
      (area);
    out;
    """
    
    try:
        response = requests.post(overpass_url, data={'data': query}, timeout=180)
        response.raise_for_status()
        data = response.json()
        
        if data['elements']:
            # Take the first matching result
            element = data['elements'][0]
            lat = element.get('lat')
            lon = element.get('lon')
            return (lat, lon)
        else:
            print(f"Address not found: {street} {housenumber}, {city}, {country}")
            return (None, None)
    except requests.exceptions.RequestException as e:
        print(f"Error querying Overpass API for address {street} {housenumber}, {city}, {country}: {e}")
        return (None, None)
    finally:
        time.sleep(pause)  # Pause to respect rate limits

def geocode_address_nominatim(street, housenumber, city, country, pause=1):
    """
    Geocode an address using the Nominatim API.

    Parameters:
        street (str): Street name.
        housenumber (str/int): House number.
        city (str): City name.
        country (str): Country name.
        pause (int): Seconds to wait between requests to respect rate limits.

    Returns:
        tuple: (latitude, longitude) if found, else (None, None)
    """
    base_url = "https://nominatim.openstreetmap.org/search?"
    address = f"{housenumber} {street}, {city}, {country}"
    params = {
        'q': address,
        'format': 'json',
        'addressdetails': 1,
        'limit': 1
    }
    url = base_url + urlencode(params)
    
    headers = {
        'User-Agent': 'YourAppName/1.0 (your.email@example.com)'  # Replace with your details
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=180)
        response.raise_for_status()
        data = response.json()
        
        if data:
            lat = float(data[0]['lat'])
            lon = float(data[0]['lon'])
            return (lat, lon)
        else:
            print(f"Address not found: {address}")
            return (None, None)
    except requests.exceptions.RequestException as e:
        print(f"Error querying Nominatim API for address {address}: {e}")
        return (None, None)
    finally:
        time.sleep(pause)  # Pause to respect rate limits

def get_distance_valhalla(origin, destination):
    """
    Get travel distance between two points using Valhalla API.
    
    Parameters:
        origin (tuple): (latitude, longitude) of origin.
        destination (tuple): (latitude, longitude) of destination.
    
    Returns:
        float: Distance in kilometers, or None if not found.
    """
    valhalla_url = 'https://valhalla1.openstreetmap.de/route'
    headers = {'Content-Type': 'application/json'}
    params = {
        "locations": [
            {"lat": origin[0], "lon": origin[1]},
            {"lat": destination[0], "lon": destination[1]}
        ],
        "costing": "auto",
        "directions_options": {"units": "kilometers"}
    }
    
    try:
        response = requests.post(valhalla_url, json=params, headers=headers, timeout=180)
        response.raise_for_status()
        data = response.json()
        distance = data['trip']['legs'][0]['summary']['length']
        return distance
    except requests.exceptions.RequestException as e:
        print(f"Error querying Valhalla API for origin {origin} and destination {destination}: {e}")
        return None
    
def create_distance_matrix_valhalla(locations):
    """
    Create a distance matrix using Valhalla API.
    
    Parameters:
        locations (list): List of (latitude, longitude) tuples.
    
    Returns:
        np.ndarray: Distance matrix.
    """
    n = len(locations)
    distance_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i + 1, n):
            distance = get_distance_valhalla(locations[i], locations[j])
            if distance is not None:
                distance_matrix[i, j] = distance
                distance_matrix[j, i] = distance
            else:
                distance_matrix[i, j] = np.inf
                distance_matrix[j, i] = np.inf
            print(f"Distance between {i} and {j}: {distance} km")
    
    return distance_matrix


### read in adress data 

In [5]:
# read address file (open document spreadsheet - but *.csv is also possible!)
adress_file = 'addresses_us.ods'
addresses_df = get_adress_data(adress_file)
# Display the first few rows
print(addresses_df.head())


                 street housenumber                  city country
0            10 Mile Rd     18600 W  Southfield, MI 48075     USA
1            10 Mile Rd     18900 W  Southfield, MI 48075     USA
2            Parsons Dr       18687  Southfield, MI 48075     USA
3  George Washington Dr       18349  Southfield, MI 48075     USA
4            Addison Dr       19135  Southfield, MI 48075     USA


### overpass map matching to get lat/lon coordinates

In [6]:
# Initialize lists to store coordinates
latitudes = []
longitudes = []

# Iterate over each address and geocode
for index, row in addresses_df.iterrows():
    street = row['street']
    housenumber = row['housenumber']
    city = row['city']
    country = row['country']
    
    #lat, lon = geocode_address_overpass(street, housenumber, city, country)
    lat, lon = geocode_address_nominatim(street, housenumber, city, country)
    latitudes.append(lat)
    longitudes.append(lon)
    
    print(f"Geocoded: {street} {housenumber}, {city} -> ({lat}, {lon})")

# Add coordinates to the DataFrame
addresses_df['latitude'] = latitudes
addresses_df['longitude'] = longitudes

# Remove addresses that couldn't be geocoded
addresses_df = addresses_df.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)

print("\nGeocoded Addresses:")
addresses_df.head()


Geocoded: 10 Mile Rd 18600 W, Southfield, MI 48075 -> (42.47334673154362, -83.22653383221476)
Geocoded: 10 Mile Rd 18900 W, Southfield, MI 48075 -> (42.47326396644295, -83.22985897986577)
Geocoded: Parsons Dr 18687, Southfield, MI 48075 -> (42.47565762334809, -83.22715727790715)
Geocoded: George Washington Dr 18349, Southfield, MI 48075 -> (42.47137000513243, -83.22419894126222)
Geocoded: Addison Dr 19135, Southfield, MI 48075 -> (42.4715727148751, -83.2326548750282)
Geocoded: Lee Baker Dr 24482, Southfield, MI 48075 -> (42.469276890427984, -83.23437837190156)

Geocoded Addresses:


,street,housenumber,city,country,latitude,longitude
0,10 Mile Rd,18600 W,"Southfield, MI 48075",USA,42.473347,-83.226534
1,10 Mile Rd,18900 W,"Southfield, MI 48075",USA,42.473264,-83.229859
2,Parsons Dr,18687,"Southfield, MI 48075",USA,42.475658,-83.227157
3,George Washington Dr,18349,"Southfield, MI 48075",USA,42.471370,-83.224199
4,Addison Dr,19135,"Southfield, MI 48075",USA,42.471573,-83.232655


### Create Distance Matrix

In [7]:
# Extract coordinates from the DataFrame
coordinates = list(zip(addresses_df['latitude'], addresses_df['longitude']))
# Create the distance matrix
distance_matrix = create_distance_matrix_valhalla(coordinates)

Distance between 0 and 1: 0.272 km
Distance between 0 and 2: 0.628 km
Distance between 0 and 3: 0.906 km
Distance between 0 and 4: 0.614 km
Distance between 0 and 5: 0.954 km
Distance between 1 and 2: 0.718 km
Distance between 1 and 3: 0.643 km
Distance between 1 and 4: 0.342 km
Distance between 1 and 5: 1.044 km
Distance between 2 and 3: 0.771 km
Distance between 2 and 4: 0.89 km
Distance between 2 and 5: 1.399 km
Distance between 3 and 4: 0.811 km
Distance between 3 and 5: 0.919 km
Distance between 4 and 5: 1.193 km


## Custom Balanced K-Means with Preferences

In [17]:
from scipy.spatial.distance import cdist
import random

def balanced_kmeans_with_preferences(X, k, preferences, max_iters=100):
    """
    Perform balanced K-Means clustering with preferences for certain pairs.
    
    Parameters:
        X (np.ndarray): Coordinates array of shape (n_samples, n_features).
        k (int): Number of clusters.
        preferences (list of tuples): List of index pairs that should be in the same cluster.
        max_iters (int): Maximum number of iterations.
    
    Returns:
        np.ndarray: Cluster labels for each point.
    """
    n = len(X)
    size_per_cluster = n // k
    labels = np.full(n, -1, dtype=int)
    
    # Initialize centroids randomly
    initial_indices = random.sample(range(n), k)
    centroids = X[initial_indices]
    
    for iteration in range(max_iters):
        #print(f"Iteration {iteration + 1}")
        
        # Compute distances from points to centroids
        distances = cdist(X, centroids, 'euclidean')
        
        # Assign labels based on closest centroid
        new_labels = np.argmin(distances, axis=1)
        
        # Initialize empty clusters
        clusters = {i: [] for i in range(k)}
        
        # Assign points to clusters ensuring balanced size
        for idx in np.argsort(new_labels):
            preferred_cluster = new_labels[idx]
            if len(clusters[preferred_cluster]) < size_per_cluster:
                clusters[preferred_cluster].append(idx)
                labels[idx] = preferred_cluster
            else:
                # Assign to the next available cluster
                for cluster_id in range(k):
                    if len(clusters[cluster_id]) < size_per_cluster:
                        clusters[cluster_id].append(idx)
                        labels[idx] = cluster_id
                        break
        
        # Handle preferences
        for (a, b) in preferences:
            if labels[a] != labels[b]:
                # Try to assign both to the same cluster if possible
                cluster_a = labels[a]
                cluster_b = labels[b]
                
                if len(clusters[cluster_a]) < size_per_cluster:
                    clusters[cluster_a].append(b)
                    clusters[cluster_b].remove(b)
                    labels[b] = cluster_a
                elif len(clusters[cluster_b]) < size_per_cluster:
                    clusters[cluster_b].append(a)
                    clusters[cluster_a].remove(a)
                    labels[a] = cluster_b
        
        # Recompute centroids
        for i in range(k):
            cluster_points = X[clusters[i]]
            if len(cluster_points) > 0:
                centroids[i] = np.mean(cluster_points, axis=0)
        
        # Check for convergence
        if np.array_equal(labels, new_labels):
            print("Convergence reached.")
            break
    
    return labels


In [18]:
import folium

def visualize_clusters_folium(df, cluster_column, output_html='children_homes_clusters.html'):
    """
    Visualize clustered locations on a Folium map.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing latitude, longitude, and cluster labels.
        cluster_column (str): Name of the column with cluster labels.
        output_html (str): Filename for the saved HTML map.
    
    Returns:
        folium.Map: Folium map object.
    """
    # Calculate the average latitude and longitude for centering the map
    avg_lat = df['latitude'].mean()
    avg_lon = df['longitude'].mean()
    
    # Initialize Folium map
    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=13)
    
    # Define colors for clusters
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue',
              'darkpurple', 'pink', 'lightblue', 'lightgreen', 'gray',
              'black', 'lightgray']
    
    # Add markers to the map
    for idx, row in df.iterrows():
        cluster_id = row[cluster_column]
        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=f"Child {idx} - Group {cluster_id}",
            icon=folium.Icon(color=colors[cluster_id % len(colors)])
        ).add_to(m)
    
    # Save the map to an HTML file
    m.save(output_html)
    print(f"Map saved to {output_html}")
    
    return m

### apply - preference child 3&5 in one group

In [19]:
# Example preferences: list of tuples with indices that should be in the same cluster
# Replace with actual index pairs based on your data
preferences = [
    (3, 5)  # Child 3 and Child 5 should be in the same cluster
]
# Convert coordinates to NumPy array
X = np.array(coordinates)

# Number of clusters
k = 2

# Perform balanced K-Means with preferences
labels = balanced_kmeans_with_preferences(X, k, preferences, max_iters=200)

# Add cluster labels to the DataFrame
addresses_df['cluster'] = labels

# Display cluster assignments
print("\nCluster Assignments:")
print(addresses_df)



Cluster Assignments:
                 street housenumber                  city country   latitude  \
0            10 Mile Rd     18600 W  Southfield, MI 48075     USA  42.473347   
1            10 Mile Rd     18900 W  Southfield, MI 48075     USA  42.473264   
2            Parsons Dr       18687  Southfield, MI 48075     USA  42.475658   
3  George Washington Dr       18349  Southfield, MI 48075     USA  42.471370   
4            Addison Dr       19135  Southfield, MI 48075     USA  42.471573   
5          Lee Baker Dr       24482  Southfield, MI 48075     USA  42.469277   

   longitude  cluster  
0 -83.226534        0  
1 -83.229859        0  
2 -83.227157        0  
3 -83.224199        1  
4 -83.232655        1  
5 -83.234378        1  


### visualization

In [ ]:
# Visualize the clusters
m = visualize_clusters_folium(addresses_df, 'cluster')

# If you're using Jupyter Notebook, display the map inline
m


Map saved to children_homes_clusters.html


In [21]:
# Example preferences: list of tuples with indices that should be in the same cluster
# Replace with actual index pairs based on your data
preferences = [
    (2, 5)  # Child 2 and Child 5 should be in the same cluster
]
# Convert coordinates to NumPy array
X = np.array(coordinates)

# Number of clusters
k = 2

# Perform balanced K-Means with preferences
labels = balanced_kmeans_with_preferences(X, k, preferences, max_iters=200)

# Add cluster labels to the DataFrame
addresses_df['cluster'] = labels

# Display cluster assignments
print("\nCluster Assignments:")
print(addresses_df)


Convergence reached.

Cluster Assignments:
                 street housenumber                  city country   latitude  \
0            10 Mile Rd     18600 W  Southfield, MI 48075     USA  42.473347   
1            10 Mile Rd     18900 W  Southfield, MI 48075     USA  42.473264   
2            Parsons Dr       18687  Southfield, MI 48075     USA  42.475658   
3  George Washington Dr       18349  Southfield, MI 48075     USA  42.471370   
4            Addison Dr       19135  Southfield, MI 48075     USA  42.471573   
5          Lee Baker Dr       24482  Southfield, MI 48075     USA  42.469277   

   longitude  cluster  
0 -83.226534        0  
1 -83.229859        1  
2 -83.227157        0  
3 -83.224199        0  
4 -83.232655        1  
5 -83.234378        1  


In [22]:
# Visualize the clusters
m = visualize_clusters_folium(addresses_df, 'cluster')

# If you're using Jupyter Notebook, display the map inline
m


Map saved to children_homes_clusters.html


... didn't worked ... work in progress!